# 01 — Exploratory Data Analysis

What is actually in `Data/`, and what does it imply for preprocessing and
training?

The questions worth answering before any model is trained:

1. **Balance** — how many images per species, and how skewed?
2. **Geometry** — what resolutions and aspect ratios, and what does that
   mean for the resize strategy?
3. **Integrity** — duplicates, corrupt files, odd colour modes.
4. **Separability** — is there colour signal, or must shape carry it?

Nothing here writes to the repository; `leaf-train prepare` owns the
manifest.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from medicinal_leaf.config.settings import load_settings
from medicinal_leaf.data.ingestion import build_index_from_settings
from medicinal_leaf.data.validation import find_duplicates, validate_index
from medicinal_leaf.preprocessing.image import load_image
from medicinal_leaf.preprocessing.segmentation import leaf_mask, mask_coverage

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 120)

settings = load_settings("development")
print(settings.data.raw_dir)

## 1. Build the index

One row per image, read from the file headers — no pixels decoded yet.

In [ ]:
df = build_index_from_settings(settings)

print(f"{len(df)} images across {df['class_name'].nunique()} classes")
df.head()

## 2. Class balance

The ratio between the largest and smallest class decides whether
`training.class_weighting` is worth enabling, and whether accuracy is a
misleading headline metric.

In [ ]:
counts = df["class_name"].value_counts().sort_values(ascending=False)
imbalance = counts.max() / counts.min()

print(counts.to_string())
print(f"\nimbalance ratio: {imbalance:.2f}x")
print(f"a constant predictor would score {counts.max() / counts.sum():.2%} accuracy")

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=counts.values, y=counts.index, hue=counts.index, legend=False, ax=ax, palette="crest")
ax.set(xlabel="images", ylabel="", title="Images per species")
for i, value in enumerate(counts.values):
    ax.text(value + 1, i, str(value), va="center")
plt.show()

## 3. Image dimensions

If most images are already square-ish, the resize strategy barely matters.
If they are not, squashing them distorts leaf shape — which is exactly the
feature the classifier should be using.

In [ ]:
print(df[["width", "height", "aspect_ratio", "size_bytes"]].describe().round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.scatterplot(data=df, x="width", y="height", hue="class_name", alpha=0.6, ax=axes[0])
limit = max(df["width"].max(), df["height"].max())
axes[0].plot([0, limit], [0, limit], ls="--", c="grey", lw=1)
axes[0].set(title="Resolution (dashed line = square)")
axes[0].legend(fontsize=8)

sns.histplot(data=df, x="aspect_ratio", bins=40, kde=True, ax=axes[1])
axes[1].axvline(1.0, ls="--", c="crimson", lw=1)
axes[1].set(title="Aspect ratio (width / height)")

plt.tight_layout()
plt.show()

In [ ]:
# How much of the frame survives a letterbox to a square?
usable = np.minimum(df["aspect_ratio"], 1 / df["aspect_ratio"])

print(f"median usable area after letterboxing: {usable.median():.1%}")
print(f"images losing more than half the frame: {(usable < 0.5).sum()}")
print(f"\nsmallest side across the set: {df[['width', 'height']].min().min()} px")
print(
    f"images smaller than the target {settings.preprocessing.image_size}px:",
    int((df[["width", "height"]].min(axis=1) < settings.preprocessing.image_size).sum()),
)

## 4. Colour modes and file sizes

Grayscale or palette images among RGB ones mean some samples carry no colour
information at all — worth knowing before trusting a colour-based cue.

In [ ]:
print(df["mode"].value_counts().to_string())
print()
print((df["size_bytes"] / 1024).describe().round(1).to_string(), "(KB)")

fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(
    data=df, x="class_name", y=df["size_bytes"] / 1024, hue="class_name", legend=False, ax=ax
)
ax.set(xlabel="", ylabel="KB", title="File size by species")
plt.show()

## 5. Integrity

The check that matters most: byte-identical images. A duplicate spanning two
splits means the test set is partly memorised, and the reported accuracy is
fiction.

In [ ]:
report = validate_index(
    df,
    min_side=settings.data.min_side,
    max_aspect_ratio=settings.data.max_aspect_ratio,
    min_images_per_class=settings.data.min_images_per_class,
    max_imbalance_ratio=settings.data.max_imbalance_ratio,
)

print(report.summary())
report.to_frame().head(20)

In [ ]:
duplicates = find_duplicates(df)
print(f"{len(duplicates)} duplicate group(s)")

for digest, paths in list(duplicates.items())[:5]:
    classes = {Path(p).parent.name for p in paths}
    flag = "  <-- ACROSS CLASSES" if len(classes) > 1 else ""
    print(f"\n{digest[:12]} ({len(paths)} copies){flag}")
    for p in paths:
        print("   ", Path(p).parent.name, "/", Path(p).name)

## 6. What the images actually look like

A sample per species. Look for background clutter, hands, scale cards,
multiple leaves in frame — anything a classifier could latch onto instead of
the leaf itself.

In [ ]:
rng = np.random.default_rng(settings.seed)
classes = sorted(df["class_name"].unique())
per_class = 4

fig, axes = plt.subplots(len(classes), per_class, figsize=(3 * per_class, 3 * len(classes)))

for row, class_name in enumerate(classes):
    subset = df[df["class_name"] == class_name]
    picks = rng.choice(
        subset["file_path"].to_numpy(), size=min(per_class, len(subset)), replace=False
    )
    for col in range(per_class):
        ax = axes[row, col]
        ax.axis("off")
        if col < len(picks):
            ax.imshow(load_image(picks[col]))
            if col == 0:
                ax.set_title(class_name, loc="left", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

## 7. Does segmentation help here?

`preprocessing.segmentation` isolates the leaf with an Excess Green index.
Check the mask on a handful of images before enabling `segment_leaf` in the
config — coverage near 0 or 1 means it found nothing useful and fell back to
the original frame.

In [ ]:
samples = df.sample(min(4, len(df)), random_state=settings.seed)
fig, axes = plt.subplots(2, len(samples), figsize=(3.2 * len(samples), 6.5))

for col, (_, row) in enumerate(samples.iterrows()):
    image = load_image(row["file_path"])
    array = np.asarray(image)
    mask = leaf_mask(array)

    axes[0, col].imshow(image)
    axes[0, col].set_title(row["class_name"], fontsize=10)
    axes[1, col].imshow(mask, cmap="gray")
    axes[1, col].set_title(f"coverage {mask_coverage(mask):.1%}", fontsize=9)
    axes[0, col].axis("off")
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()

## 8. Is there colour signal?

Mean RGB per species, on thumbnails. Well-separated means colour alone
carries some signal; overlapping means shape and texture have to do the
work — and that argues for aspect-preserving resizing and stronger
geometric augmentation.

In [ ]:
sample = df.groupby("class_name", group_keys=False).sample(
    n=min(40, int(df["class_name"].value_counts().min())), random_state=settings.seed
)

rows = []
for _, row in sample.iterrows():
    thumbnail = load_image(row["file_path"]).resize((32, 32))
    mean_rgb = np.asarray(thumbnail).reshape(-1, 3).mean(axis=0)
    rows.append(
        {"class_name": row["class_name"], "R": mean_rgb[0], "G": mean_rgb[1], "B": mean_rgb[2]}
    )

colour = pd.DataFrame(rows)
colour["greenness"] = 2 * colour["G"] - colour["R"] - colour["B"]

print(colour.groupby("class_name")[["R", "G", "B", "greenness"]].mean().round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.scatterplot(data=colour, x="R", y="G", hue="class_name", alpha=0.7, ax=axes[0])
axes[0].set(title="Mean red vs green")
sns.kdeplot(data=colour, x="greenness", hue="class_name", fill=True, alpha=0.3, ax=axes[1])
axes[1].set(title="Excess green distribution")
plt.tight_layout()
plt.show()

## Takeaways

Fill these in from the output above — they are the inputs to the config:

| Observation | Config consequence |
| --- | --- |
| Imbalance ratio | `training.class_weighting`, and macro-F1 over accuracy |
| Aspect ratio spread | `preprocessing.resize_strategy` |
| Smallest images | `preprocessing.image_size`, `data.min_side` |
| Duplicate groups | must be resolved before `prepare` will proceed |
| Background clutter | `preprocessing.segment_leaf` |
| Colour overlap | augmentation strength, especially hue jitter |

Next: `leaf-train prepare`, then `leaf-train fit`.